# 第16章：自定义扩展开发

## 本章学习目标

- 实现自定义数据提供器
- 开发自定义模型
- 创建自定义策略
- 接入 qlib 体系

---

## 16.1 扩展开发概述

Qlib 的模块化设计使得用户可以轻松扩展各个组件。

### 可扩展组件

```
Qlib 扩展点
├── 数据层
│   ├── DataProvider: 数据提供器
│   ├── DataHandler: 数据处理器
│   └── Processor: 数据预处理器
├── 模型层
│   └── Model: 预测模型
└── 策略层
    └── Strategy: 交易策略
```

In [ ]:
import qlib
from qlib.model.base import Model
from qlib.strategy.base import BaseStrategy
from qlib.data.dataset.handler import DataHandlerLP
import pandas as pd
import numpy as np

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 16.2 自定义数据处理器

In [ ]:
# 自定义 DataHandler
from qlib.data.dataset.handler import DataHandlerLP

class CustomAlphaHandler(DataHandlerLP):
    """
    自定义因子处理器
    
    包含一组自定义的技术因子
    """
    
    def __init__(
        self,
        instruments="csi300",
        start_time=None,
        end_time=None,
        freq="day",
        infer_processors=None,
        learn_processors=None,
        **kwargs
    ):
        # 定义因子表达式
        self.fields = [
            # 价格因子
            "$close",
            "$volume",
            "$vwap",
            
            # 动量因子
            "$close / Ref($close, 5) - 1",
            "$close / Ref($close, 10) - 1",
            "$close / Ref($close, 20) - 1",
            
            # 波动率因子
            "Std($close / Ref($close, 1) - 1, 5)",
            "Std($close / Ref($close, 1) - 1, 10)",
            "Std($close / Ref($close, 1) - 1, 20)",
            
            # 成交量因子
            "$volume / Mean($volume, 5)",
            "$volume / Mean($volume, 20)",
            
            # 技术因子
            "($close - Min($low, 14)) / (Max($high, 14) - Min($low, 14))",
            "($close - Mean($close, 20)) / Mean($close, 20)",
        ]
        
        super().__init__(
            instruments=instruments,
            start_time=start_time,
            end_time=end_time,
            freq=freq,
            infer_processors=infer_processors,
            learn_processors=learn_processors,
            **kwargs
        )

# 测试自定义 Handler
handler = CustomAlphaHandler(
    instruments="csi300",
    start_time="2022-01-01",
    end_time="2022-12-31",
)

df = handler.fetch()
print(f"自定义 Handler 数据形状: {df.shape}")
print(f"因子数量: {len(df.columns)}")

## 16.3 自定义模型

In [ ]:
# 自定义模型
from qlib.model.base import Model
from sklearn.ensemble import GradientBoostingRegressor

class CustomGBDTModel(Model):
    """
    自定义 GBDT 模型
    
    封装 sklearn 的 GradientBoostingRegressor
    """
    
    def __init__(self, 
                 n_estimators=100,
                 max_depth=5,
                 learning_rate=0.1,
                 random_state=42):
        self.n_estimators = n_estimators
        max_depth = max_depth
        self.learning_rate = learning_rate
        self.random_state = random_state
        
        self.model = None
    
    def fit(self, dataset):
        """训练模型"""
        df_train = dataset.prepare("train")
        
        X = df_train['feature'].values
        y = df_train['label'].values.ravel()
        
        # 处理缺失值
        mask = ~np.isnan(y)
        X = X[mask]
        y = y[mask]
        
        # 训练
        self.model = GradientBoostingRegressor(
            n_estimators=self.n_estimators,
            max_depth=5,
            learning_rate=self.learning_rate,
            random_state=self.random_state,
        )
        self.model.fit(X, y)
        
        return self
    
    def predict(self, dataset):
        """预测"""
        df_test = dataset.prepare("test")
        X = df_test['feature'].values
        
        pred = self.model.predict(X)
        
        return pd.Series(pred, index=df_test.index)
    
    def save(self, path):
        """保存模型"""
        import pickle
        with open(path, "wb") as f:
            pickle.dump(self.model, f)
    
    def load(self, path):
        """加载模型"""
        import pickle
        with open(path, "rb") as f:
            self.model = pickle.load(f)
        return self

print("CustomGBDTModel 定义完成")

## 16.4 自定义策略

In [ ]:
# 自定义策略
from qlib.strategy.base import BaseStrategy
from qlib.backtest.decision import TradeDecisionWO, Order

class CustomTopkStrategy(BaseStrategy):
    """
    自定义 Topk 策略
    
    根据信号选择 topk 只股票
    """
    
    def __init__(self, signal, topk=30, risk_degree=0.95):
        """
        参数:
            signal: 预测信号 Series
            topk: 持仓数量
            risk_degree: 仓位比例
        """
        self.signal = signal
        self.topk = topk
        self.risk_degree = risk_degree
        super().__init__()
    
    def get_topk_stocks(self, date):
        """获取指定日期的 topk 股票"""
        try:
            signal_day = self.signal.xs(date, level='datetime')
            topk_stocks = signal_day.nlargest(self.topk).index.tolist()
            return topk_stocks
        except KeyError:
            return []
    
    def generate_trade_decision(self, execute_result=None):
        """生成交易决策"""
        # 获取当前日期
        trade_step = self.trade_calendar.get_trade_step()
        
        # 获取 topk 股票
        topk_stocks = self.get_topk_stocks(trade_step)
        
        # 创建订单列表
        order_list = []
        
        # ... 具体的交易逻辑
        
        return TradeDecisionWO(order_list, self)

print("CustomTopkStrategy 定义完成")

## 16.5 注册自定义组件

In [ ]:
# 使用 YAML 配置注册自定义组件
custom_config = """
# 自定义组件配置示例

qlib_init:
    provider_uri: ~/.qlib/qlib_data/cn_data
    region: cn

task:
    model:
        class: CustomGBDTModel
        module_path: my_module.models  # 自定义模块路径
        kwargs:
            n_estimators: 200
            learning_rate: 0.05
    
    dataset:
        class: DatasetH
        module_path: qlib.data.dataset
        kwargs:
            handler:
                class: CustomAlphaHandler
                module_path: my_module.handlers  # 自定义模块路径
                kwargs:
                    instruments: csi300
                    start_time: 2018-01-01
                    end_time: 2022-12-31
"""

print("自定义组件 YAML 配置示例:")
print(custom_config)

## 16.6 开发最佳实践

In [ ]:
# 开发最佳实践
print("自定义组件开发最佳实践:")
print("=" * 60)

best_practices = {
    "代码组织": {
        "描述": "将自定义组件放在单独的模块中",
        "示例": "my_module/models.py, my_module/handlers.py",
    },
    "接口规范": {
        "描述": "严格遵循 qlib 的接口规范",
        "示例": "Model.fit(), Model.predict(), Strategy.generate_trade_decision()",
    },
    "参数验证": {
        "描述": "在初始化时验证参数",
        "示例": "检查参数范围和类型",
    },
    "错误处理": {
        "描述": "提供有意义的错误信息",
        "示例": "使用 try-except 捕获异常",
    },
    "文档": {
        "描述": "添加详细的文档字符串",
        "示例": "使用 Google/Numpy 风格的文档",
    },
    "单元测试": {
        "描述": "编写单元测试",
        "示例": "使用 pytest 编写测试用例",
    },
}

for practice, details in best_practices.items():
    print(f"\n{practice}:")
    print(f"  描述: {details['描述']}")
    print(f"  示例: {details['示例']}")

## 16.7 实践练习

In [ ]:
# 练习1: 实现一个自定义预处理器
# 功能：Winsorization (去极值)

# 你的代码



# 参考答案
# from qlib.data.dataset.processor import Processor
# class WinsorizeProcessor(Processor):
#     def __init__(self, fields_group="feature", n_std=3):
#         super().__init__(fields_group=fields_group)
#         self.n_std = n_std
#     
#     def __call__(self, df):
#         mean = df.mean()
#         std = df.std()
#         for col in df.columns:
#             df[col] = df[col].clip(mean[col] - self.n_std * std[col],
#                                    mean[col] + self.n_std * std[col])

In [ ]:
# 练习2: 实现一个自定义因子
# 功能：计算自定义技术指标

# 你的代码



# 提示：在 CustomAlphaHandler 中添加新的因子表达式

In [ ]:
# 练习3: 实现一个自定义模型评估器
# 功能：计算 IC、Rank IC、ICIR 等指标

# 你的代码



# 提示：封装 evaluate_predictions 函数

## 16.8 本章小结

本章我们学习了：

1. **自定义数据处理器**：
   - 继承 DataHandlerLP
   - 定义因子表达式

2. **自定义模型**：
   - 继承 Model 基类
   - 实现 fit/predict 接口

3. **自定义策略**：
   - 继承 BaseStrategy
   - 实现 generate_trade_decision

4. **组件注册**：
   - 通过 YAML 配置使用自定义组件

### 下一部分预告

下一部分我们将进行综合实战项目，包括：
- 多因子选股策略实战
- 行业轮动策略实战